# TartanIMU Challenge — unified context model (v1) → `submission_v1.csv`

**Task:** predict the mean 3-D body-frame velocity of every 1 s window (200 Hz, 6-axis IMU) with **one model** shared across
car / dog / drone / human. Score = `0.6·AVE/0.7356 + 0.4·ATE20/3.116`, macro-averaged over platforms (lower is better).

**Approach (v1):**
1. **Context, not isolated windows** — test trajectories are given whole and windows are contiguous, so the network reads
   16 s chunks (past *and* future) and predicts every window in the chunk; at inference chunks slide with overlap and are
   Hann-weight averaged.
2. **Unified network, implicit embodiment** — no platform input; an auxiliary platform-classification head makes the shared
   features embodiment-aware without any routing.
3. **Metric-aware training** — platform-balanced sampling (macro average), vector-Huber window loss, dense 20 Hz loss and an
   integrated-error ("drift") loss mirroring ATE20's sensitivity to correlated bias.
4. **Physically consistent augmentation** — small random sensor-mount rotations applied to IMU *and* target velocity,
   accel/gyro bias, scale and white noise.

Architecture: strided conv stem (200 Hz → 20 Hz tokens) → 8 dilated depthwise TCN blocks (~13 s receptive field) →
2-layer transformer encoder → per-token velocity head (1.78 M params). EMA weights, OneCycle LR, 30 epochs × 250 steps.

**Result:** val 0.2247 (car 0.171 / dog 0.134 / drone 0.453 / human 0.140) → public LB 0.378.
Works on Kaggle (CUDA), Apple Silicon (MPS) or CPU. Set `QUICK = True` for a 2-minute smoke run.

In [ ]:
import math, time, json
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------------------------------------------------------- paths / config
_cands = [Path("/kaggle/input/competitions/tartan-imu-challenge-iros2026"), Path("/kaggle/input/tartan-imu-challenge-iros2026"),
          Path("data"), Path("../data")]
DATA = next(p for p in _cands if (p / "index" / "train_windows.csv").exists())
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
print("data:", DATA.resolve(), "| out:", OUT.resolve())

QUICK = False                 # True -> tiny run to check the pipeline end to end
CFG = dict(epochs=2 if QUICK else 30, steps=20 if QUICK else 250, batch=64, lr=1.5e-3, wd=0.02, chunk=16, width=128,
           seed=0, eval_every=1 if QUICK else 2, rot_deg=15.0)

WIN, TOK = 200, 20            # samples per window, tokens per window (200 Hz -> 20 Hz)
PLATFORMS = ["car", "dog", "drone", "human"]
PLAT2ID = {p: i for i, p in enumerate(PLATFORMS)}
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("device:", device)

## 1. Data

In [ ]:
def read_index(split):
    df = pd.read_csv(DATA / "index" / f"{split}_windows.csv").dropna(axis=1, how="all")
    if split != "test":
        df = df.merge(pd.read_csv(DATA / "index" / f"{split}_targets.csv"), on="window_id")
    return df.sort_values(["traj_id", "win_idx"]).reset_index(drop=True)


def traj_path(split, traj_id):
    return DATA / "test" / f"{traj_id}.npz" if split == "test" else DATA / split / traj_id.split("_")[0] / f"{traj_id}.npz"


def load_split(split, keys=("imu",)):
    """{traj_id: {key: array, 'n_win': int}} — whole trajectories in memory (imu = [ax,ay,az,gx,gy,gz])."""
    out = {}
    for tid in read_index(split)["traj_id"].unique():
        with np.load(traj_path(split, tid)) as d:
            out[tid] = {k: d[k] for k in keys if k in d.files}
            out[tid]["n_win"] = len(d["ts"]) // WIN
    return out


train_idx, val_idx, test_idx = read_index("train"), read_index("val"), read_index("test")
print({s: (len(df), df.traj_id.nunique()) for s, df in [("train", train_idx), ("val", val_idx), ("test", test_idx)]})
train_idx.groupby("platform").size()

## 2. Official metric (TartanIMU score) + val self-scoring

In [ ]:
# --- verbatim from the organisers' starter kit: starter/kaggle_metric_tartanimu_score.py ---
SEGMENT_LENGTH_M, MIN_SEGMENT_POINTS = 20.0, 3
W_AVE, W_ATE, AVE_REF, ATE_REF = 0.6, 0.4, 0.7356384388, 3.1160277267


def _quat_to_R(q):
    x, y, z, w = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
    n = np.sqrt(x * x + y * y + z * z + w * w); n[n == 0] = 1.0
    x, y, z, w = x / n, y / n, z / n, w / n
    R = np.empty((q.shape[0], 3, 3))
    R[:, 0, 0] = 1 - 2 * (y * y + z * z); R[:, 0, 1] = 2 * (x * y - z * w); R[:, 0, 2] = 2 * (x * z + y * w)
    R[:, 1, 0] = 2 * (x * y + z * w); R[:, 1, 1] = 1 - 2 * (x * x + z * z); R[:, 1, 2] = 2 * (y * z - x * w)
    R[:, 2, 0] = 2 * (x * z - y * w); R[:, 2, 1] = 2 * (y * z + x * w); R[:, 2, 2] = 1 - 2 * (x * x + y * y)
    return R


def _umeyama_align(P, Q):
    muP, muQ = P.mean(0), Q.mean(0)
    H = (P - muP).T @ (Q - muQ)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    return R, muQ - R @ muP


def _segment_bounds(Q, seg_len=SEGMENT_LENGTH_M, min_pts=MIN_SEGMENT_POINTS):
    cum = np.concatenate([[0.0], np.cumsum(np.linalg.norm(np.diff(Q, axis=0), axis=1))])
    cuts, reached = [], 0.0
    for i, c in enumerate(cum):
        if c >= reached + seg_len:
            cuts.append(i); reached = c
    if not cuts or cuts[-1] != len(Q) - 1:
        cuts.append(len(Q) - 1)
    segs, start = [], 0
    for end in cuts:
        if end > start:
            segs.append([start, end]); start = end
    if not segs:
        return []
    merged = [segs[0]]
    for s, e in segs[1:]:
        if e - s + 1 < min_pts:
            merged[-1][1] = e
        else:
            merged.append([s, e])
    if merged[0][1] - merged[0][0] + 1 < min_pts and len(merged) > 1:
        merged[1][0] = merged[0][0]; merged.pop(0)
    return [(s, e) for s, e in merged if e - s + 1 >= min_pts]


def _ate_segment(R, v, dt, Q):
    P = np.cumsum(np.einsum("mij,mj->mi", R, v) * dt, axis=0)
    Rr, t = _umeyama_align(P, Q)
    return float(np.sqrt(np.mean(np.sum((P @ Rr.T + t - Q) ** 2, axis=1))))


def _ate_traj(g):
    Q = g[["gx", "gy", "gz"]].to_numpy(float)
    bounds = _segment_bounds(Q)
    if not bounds:
        return float("nan")
    R = _quat_to_R(g[["qx", "qy", "qz", "qw"]].to_numpy(float))
    v = g[["vx_pred", "vy_pred", "vz_pred"]].to_numpy(float); dt = g["dt"].to_numpy(float)[:, None]
    return float(np.mean([_ate_segment(R[s:e + 1], v[s:e + 1], dt[s:e + 1], Q[s:e + 1]) for s, e in bounds]))


def _ave_traj(g):
    err = g[["vx_pred", "vy_pred", "vz_pred"]].to_numpy(float) - g[["vx_gt", "vy_gt", "vz_gt"]].to_numpy(float)
    return float(np.mean(np.linalg.norm(err, axis=1)))


def tartan_score(solution, submission):
    """Returns (score, per-platform DataFrame with ave / ate20 / score)."""
    m = solution.merge(submission.rename(columns={"vx": "vx_pred", "vy": "vy_pred", "vz": "vz_pred"}), on="window_id", how="left")
    assert not m[["vx_pred", "vy_pred", "vz_pred"]].isnull().any().any(), "missing window_id rows"
    recs = []
    for tid, g in m.groupby("traj_id", sort=False):
        g = g.sort_values("win_idx")
        recs.append({"platform": g["platform"].iloc[0], "ate20": _ate_traj(g), "ave": _ave_traj(g)})
    pp = pd.DataFrame(recs).dropna().groupby("platform")[["ave", "ate20"]].mean()
    pp["score"] = W_AVE * pp["ave"] / AVE_REF + W_ATE * pp["ate20"] / ATE_REF
    return W_AVE * pp["ave"].mean() / AVE_REF + W_ATE * pp["ate20"].mean() / ATE_REF, pp


def build_solution(split):
    """Scorer's solution frame: quaternion at window mid-sample, GT position at window end, dt = window duration."""
    idx, rows = read_index(split), []
    for tid, g in idx.groupby("traj_id", sort=False):
        with np.load(traj_path(split, tid)) as d:
            quat, pos, ts, fs = d["quat"], d["pos"], d["ts"], float(d["fs"])
        w = g["win_idx"].to_numpy(); s, e, mid = w * WIN, w * WIN + WIN - 1, w * WIN + WIN // 2
        rows.append(pd.DataFrame({"window_id": g["window_id"].to_numpy(), "traj_id": tid, "win_idx": w, "platform": g["platform"].to_numpy(),
                                  "qx": quat[mid, 0], "qy": quat[mid, 1], "qz": quat[mid, 2], "qw": quat[mid, 3],
                                  "gx": pos[e, 0], "gy": pos[e, 1], "gz": pos[e, 2], "dt": ts[e] - ts[s] + 1.0 / fs,
                                  "vx_gt": g["vx"].to_numpy(), "vy_gt": g["vy"].to_numpy(), "vz_gt": g["vz"].to_numpy()}))
    return pd.concat(rows, ignore_index=True)


val_sol = build_solution("val")
zero = val_sol[["window_id"]].assign(vx=0.0, vy=0.0, vz=0.0)
gt = val_sol[["window_id"]].assign(vx=val_sol.vx_gt, vy=val_sol.vy_gt, vz=val_sol.vz_gt)
print("val sanity — all-zero: %.4f | ground truth: %.4f" % (tartan_score(val_sol, zero)[0], tartan_score(val_sol, gt)[0]))

## 3. Model

In [ ]:
IN_SCALE = torch.tensor([3.0, 3.0, 3.0, 0.4, 0.4, 0.4]).view(1, 6, 1)   # raw units -> O(1); zero-mean keeps rotations exact


class TCNBlock(nn.Module):
    def __init__(self, w, dilation, drop=0.1):
        super().__init__()
        self.norm = nn.GroupNorm(8, w)
        self.dw = nn.Conv1d(w, w, 5, padding=2 * dilation, dilation=dilation, groups=w, bias=False)
        self.pw = nn.Sequential(nn.Conv1d(w, 3 * w, 1), nn.GELU(), nn.Dropout(drop), nn.Conv1d(3 * w, w, 1))

    def forward(self, x):
        return x + self.pw(self.dw(self.norm(x)))


class IMUNet(nn.Module):
    """raw IMU chunk (B, 6, L) -> dense 20 Hz body velocity (B, L/10, 3) + platform logits (aux only)."""

    def __init__(self, width=128, blocks=8, ctx_layers=2, drop=0.1, max_tok=4096):
        super().__init__()
        self.register_buffer("in_scale", IN_SCALE.clone())
        self.stem = nn.Sequential(
            nn.Conv1d(6, width // 2, 9, stride=2, padding=4, bias=False), nn.GroupNorm(8, width // 2), nn.GELU(),
            nn.Conv1d(width // 2, width, 11, stride=5, padding=5, bias=False), nn.GroupNorm(8, width), nn.GELU())
        dil = (1, 2, 4, 8, 16, 32, 1, 2)
        self.tcn = nn.Sequential(*[TCNBlock(width, dil[i % len(dil)], drop) for i in range(blocks)])
        self.pos = nn.Parameter(torch.zeros(1, max_tok, width)); nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(width, 4, 3 * width, dropout=drop, activation="gelu", batch_first=True, norm_first=True)
        self.ctx = nn.TransformerEncoder(layer, ctx_layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 128), nn.GELU(), nn.Linear(128, 3))
        self.attn = nn.Linear(width, 1)
        self.plat = nn.Sequential(nn.LayerNorm(2 * width), nn.Linear(2 * width, 64), nn.GELU(), nn.Linear(64, 4))

    def forward(self, x):
        h = self.tcn(self.stem(x / self.in_scale)).transpose(1, 2)      # (B, N, W) at 20 Hz
        h = self.ctx(h + self.pos[:, : h.shape[1]])
        dense = self.head(h)
        a = torch.softmax(self.attn(h), dim=1)
        return dense, self.plat(torch.cat([(h * a).sum(1), h.mean(1)], dim=-1))

    @staticmethod
    def to_windows(dense):                                               # (B, N, 3) -> (B, N/TOK, 3)
        B, N, _ = dense.shape
        return dense.view(B, N // TOK, TOK, 3).mean(2)


@torch.no_grad()
def predict_trajectory(model, imu, chunk_win=16, stride_win=2, batch=32):
    """Sliding-chunk inference over a whole trajectory -> (n_win, 3). Overlaps are Hann-weighted; short trajectories edge-padded."""
    n_win = len(imu) // WIN
    x = imu[: n_win * WIN].astype(np.float32)
    pad = max(0, chunk_win - n_win)
    if pad:
        x = np.concatenate([x, np.repeat(x[-1:], pad * WIN, axis=0)])
    total = n_win + pad
    starts = list(range(0, total - chunk_win + 1, stride_win))
    if starts[-1] != total - chunk_win:
        starts.append(total - chunk_win)
    w = (0.5 - 0.5 * np.cos(2 * np.pi * (np.arange(chunk_win) + 0.5) / chunk_win)).astype(np.float32) + 0.05
    acc, wsum = np.zeros((total, 3), np.float32), np.zeros((total, 1), np.float32)
    for i in range(0, len(starts), batch):
        ss = starts[i:i + batch]
        xb = torch.from_numpy(np.stack([x[s * WIN:(s + chunk_win) * WIN].T for s in ss])).to(device)
        vb = model.to_windows(model(xb)[0]).float().cpu().numpy()
        for s, v in zip(ss, vb):
            acc[s:s + chunk_win] += v * w[:, None]; wsum[s:s + chunk_win] += w[:, None]
    return (acc / wsum)[:n_win]


model = IMUNet(width=CFG["width"]).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

## 4. Training data, sampler, augmentation, losses

In [ ]:
T = CFG["chunk"]
trajs = []
for tid, d in load_split("train", keys=("imu", "vel_body")).items():
    n = d["n_win"]; vb = d["vel_body"][: n * WIN]
    trajs.append({"id": tid, "imu": d["imu"][: n * WIN].astype(np.float32), "n_win": n, "plat": PLAT2ID[tid.split("_")[0]],
                  "win_v": vb.reshape(n, WIN, 3).mean(1).astype(np.float32),
                  "dense_v": vb.reshape(n * TOK, WIN // TOK, 3).mean(1).astype(np.float32)})
by_plat = {i: [k for k, t in enumerate(trajs) if t["plat"] == i] for i in range(4)}
plat_w = {i: np.array([trajs[k]["n_win"] for k in ks], float) for i, ks in by_plat.items()}
plat_w = {i: w / w.sum() for i, w in plat_w.items()}
rng = np.random.default_rng(CFG["seed"])
val_trajs = load_split("val", keys=("imu",))


def sample_batch(B):
    """Platform-balanced random chunks of T windows (edge-padded + masked if the trajectory is shorter)."""
    X = np.empty((B, T * WIN, 6), np.float32); Yw = np.empty((B, T, 3), np.float32)
    Yd = np.empty((B, T * TOK, 3), np.float32); M = np.ones((B, T), np.float32); P = np.empty(B, np.int64)
    for b in range(B):
        pl = rng.integers(4)
        t = trajs[rng.choice(by_plat[pl], p=plat_w[pl])]; n = t["n_win"]
        if n >= T:
            s = rng.integers(0, n - T + 1)
            X[b] = t["imu"][s * WIN:(s + T) * WIN]; Yw[b] = t["win_v"][s:s + T]; Yd[b] = t["dense_v"][s * TOK:(s + T) * TOK]
        else:
            X[b, : n * WIN] = t["imu"]; X[b, n * WIN:] = t["imu"][-1]
            Yw[b, :n] = t["win_v"]; Yw[b, n:] = 0; Yd[b, : n * TOK] = t["dense_v"]; Yd[b, n * TOK:] = 0; M[b, n:] = 0
        P[b] = t["plat"]
    return X, Yw, Yd, M, P


def rand_rotation(B, max_deg):
    """(B,3,3) rotations: uniform random axis, angle ~ U(0, max_deg) — a random sensor re-mount."""
    axis = torch.randn(B, 3, device=device); axis = axis / axis.norm(dim=1, keepdim=True)
    ang = torch.rand(B, 1, device=device) * math.radians(max_deg)
    K = torch.zeros(B, 3, 3, device=device)
    K[:, 0, 1], K[:, 0, 2], K[:, 1, 0] = -axis[:, 2], axis[:, 1], axis[:, 2]
    K[:, 1, 2], K[:, 2, 0], K[:, 2, 1] = -axis[:, 0], -axis[:, 1], axis[:, 0]
    I = torch.eye(3, device=device).expand(B, 3, 3)
    s, c = ang.sin().view(B, 1, 1), ang.cos().view(B, 1, 1)
    return I + s * K + (1 - c) * (K @ K)


def augment(X, Yw, Yd):
    """Physically consistent: rotate IMU *and* targets by the same R; then bias / scale / white noise."""
    B = X.shape[0]; R = rand_rotation(B, CFG["rot_deg"])
    acc, gyr = X[..., :3] @ R.transpose(1, 2), X[..., 3:] @ R.transpose(1, 2)
    Yw, Yd = Yw @ R.transpose(1, 2), Yd @ R.transpose(1, 2)
    acc = acc * (1 + 0.02 * torch.randn(B, 1, 3, device=device)) + 0.15 * torch.randn(B, 1, 3, device=device)
    gyr = gyr * (1 + 0.02 * torch.randn(B, 1, 3, device=device)) + 0.02 * torch.randn(B, 1, 3, device=device)
    acc = acc + 0.05 * torch.randn_like(acc); gyr = gyr + 0.004 * torch.randn_like(gyr)
    return torch.cat([acc, gyr], -1), Yw, Yd


def vhuber(pred, tgt, beta=0.25, mask=None):
    e = torch.linalg.vector_norm(pred - tgt, dim=-1)
    l = torch.where(e < beta, 0.5 * e.square() / beta, e - 0.5 * beta)
    return l.mean() if mask is None else (l * mask).sum() / mask.sum().clamp(min=1)


def evaluate(m):
    m.eval(); preds = []
    for tid, g in val_idx.groupby("traj_id", sort=False):
        v = predict_trajectory(m, val_trajs[tid]["imu"], chunk_win=T, stride_win=4)
        preds.append(pd.DataFrame({"window_id": g["window_id"].to_numpy(), "vx": v[g.win_idx, 0], "vy": v[g.win_idx, 1], "vz": v[g.win_idx, 2]}))
    m.train()
    return tartan_score(val_sol, pd.concat(preds, ignore_index=True))

## 5. Train (EMA weights, OneCycle LR, official-metric validation every 2 epochs)

In [ ]:
ema = deepcopy(model).eval()
for q in ema.parameters():
    q.requires_grad_(False)
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"], betas=(0.9, 0.99))
total_steps = CFG["epochs"] * CFG["steps"]
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=total_steps, pct_start=0.08, div_factor=20, final_div_factor=200)
CKPT = OUT / "best_v1.pt"
log, best, t0, step = [], float("inf"), time.time(), 0
for ep in range(1, CFG["epochs"] + 1):
    model.train(); tot = dict(win=0.0, dense=0.0, drift=0.0, plat=0.0)
    for it in range(CFG["steps"]):
        X, Yw, Yd, M, P = sample_batch(CFG["batch"])
        X, Yw, Yd = (torch.from_numpy(a).to(device) for a in (X, Yw, Yd))
        M, P = torch.from_numpy(M).to(device), torch.from_numpy(P).to(device)
        X, Yw, Yd = augment(X, Yw, Yd)
        dense, plat = model(X.transpose(1, 2))
        pw = model.to_windows(dense)
        l_win = vhuber(pw, Yw, mask=M)
        l_dense = vhuber(dense, Yd, mask=M.repeat_interleave(TOK, dim=1))
        cum = torch.cumsum((pw - Yw) * M[..., None], dim=1)                       # integrated body-frame error (ATE proxy)
        l_drift = (torch.linalg.vector_norm(cum, dim=-1) / torch.sqrt(torch.arange(1, T + 1, device=device))).mean()
        l_plat = F.cross_entropy(plat, P)
        loss = l_win + 0.5 * l_dense + 0.2 * l_drift + 0.05 * l_plat
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        opt.step(); sched.step(); step += 1
        with torch.no_grad():                                                     # EMA of weights
            d = min(0.998, (1 + step) / (10 + step))
            for pe, pm in zip(ema.parameters(), model.parameters()):
                pe.mul_(d).add_(pm.detach(), alpha=1 - d)
        for k, v in zip(tot, (l_win, l_dense, l_drift, l_plat)):
            tot[k] += v.item() / CFG["steps"]
    rec = {"epoch": ep, "min": round((time.time() - t0) / 60, 1), **{f"l_{k}": round(v, 4) for k, v in tot.items()}}
    if ep % CFG["eval_every"] == 0 or ep == CFG["epochs"]:
        s, pp = evaluate(ema)
        rec["val"] = round(s, 4); rec.update({f"{k}": round(v, 3) for k, v in pp["score"].items()})
        if s < best:
            best = s; torch.save({"model": ema.state_dict(), "cfg": CFG, "val_score": float(s), "epoch": ep}, CKPT)
    log.append(rec); print(json.dumps(rec))
print("best val score:", round(best, 4))
pd.DataFrame(log)

## 6. Validation breakdown of the best checkpoint (denser overlap: stride 2)

In [ ]:
ema.load_state_dict(torch.load(CKPT, map_location=device, weights_only=False)["model"]); ema.eval()


def predict_split(split, trajs_imu, idx):
    out = []
    for tid, g in idx.groupby("traj_id", sort=False):
        v = predict_trajectory(ema, trajs_imu[tid]["imu"], chunk_win=T, stride_win=2)
        w = g["win_idx"].to_numpy()
        out.append(pd.DataFrame({"window_id": g["window_id"].to_numpy(), "vx": v[w, 0], "vy": v[w, 1], "vz": v[w, 2]}))
    return pd.concat(out, ignore_index=True)


val_score, per_platform = tartan_score(val_sol, predict_split("val", val_trajs, val_idx))
print(f"val TartanIMU score: {val_score:.4f}")
per_platform.round(3)

## 7. Test inference → `submission_v1.csv`

In [ ]:
test_pred = predict_split("test", load_split("test", keys=("imu",)), test_idx)
sub = pd.read_csv(DATA / "sample_submission.csv")[["window_id"]].merge(test_pred, on="window_id", how="left")
assert len(sub) == 30644 and not sub.isna().any().any()
sub.to_csv(OUT / "submission_v1.csv", index=False)
sub.to_csv(OUT / "submission.csv", index=False)         # Kaggle picks up submission.csv from the working dir
speed = np.linalg.norm(sub[["vx", "vy", "vz"]].to_numpy(), axis=1)
print(f"wrote {OUT / 'submission_v1.csv'}: {len(sub)} rows | speed median {np.median(speed):.3f}, p99 {np.quantile(speed, .99):.3f}, max {speed.max():.3f}")
sub.head()